# Synthetic Wound Generator for Azure ML Studio

This notebook generates temporal wound healing/worsening sequences using Stable Diffusion 2.1.
Run this notebook on an **Azure ML Compute Instance** with GPU (e.g., Standard_NC6).


## 1. Environment Setup
Install required libraries. This only needs to be run once per compute instance.

In [ ]:
%pip install diffusers transformers accelerate safetensors pillow torch torchvision

## 2. Check GPU Availability
Ensure we are running on a GPU.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected. Please change Compute Instance to a GPU size (e.g., Standard_NC6).")

## 3. Load Stable Diffusion Pipeline
Loading Stable Diffusion 2.1 (optimized for float16 on GPU).

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

model_id = "stabilityai/stable-diffusion-2-1"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model: {model_id}...")
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
)
pipe = pipe.to(device)

# Enable memory slicing for lower VRAM usage
if device == "cuda":
    pipe.enable_attention_slicing()

print("✅ Pipeline loaded successfully!")

## 4. Prepare Baseline Images
Place your baseline wound images (Day 0) in a folder named `baseline_images` in the same directory as this notebook.
If you haven't uploaded them yet, you can create the folder and manually upload images via the Azure ML Studio UI.

In [ ]:
import os
from pathlib import Path

# Create directories
baseline_dir = Path("baseline_images")
output_dir = Path("synthetic_wounds")

baseline_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

print(f"Make sure your input images are in: {baseline_dir.absolute()}")
print(f"Results will be saved to: {output_dir.absolute()}")

In [ ]:
# Verify images exist
image_exts = [".jpg", ".jpeg", ".png"]
baseline_images = [f for f in baseline_dir.iterdir() if f.suffix.lower() in image_exts]

print(f"Found {len(baseline_images)} baseline images.")
if len(baseline_images) == 0:
    print("⚠️ Please upload images to the 'baseline_images' folder before proceeding.")

## 5. Define Prompts
Medical-specific prompts for healing vs. worsening progressions.

In [ ]:
HEALING_PROMPTS = {
    "day_7": "medical wound photo, 15% smaller wound, increased pink granulation tissue, reduced exudate, healthier wound bed, clinical photography",
    "day_14": "medical wound photo, 35% smaller wound, epithelial tissue forming at edges, pink healthy granulation, reduced inflammation, clinical photography",
    "day_21": "medical wound photo, 60% smaller wound, re-epithelialization progressing, healthy pink tissue, minimal exudate, healing margins, clinical photography"
}

WORSENING_PROMPTS = {
    "day_7": "medical wound photo, 10% larger wound, increased slough tissue, yellow exudate, inflamed surrounding skin, clinical photography",
    "day_14": "medical wound photo, 25% larger wound, necrotic tissue present, purulent exudate, macerated edges, clinical photography",
    "day_21": "medical wound photo, 40% larger wound, significant necrosis, heavy purulent drainage, erythematous surrounding tissue, clinical photography"
}

NEGATIVE_PROMPT = "cartoon, drawing, illustration, text, watermark, low quality, blurry, distorted, unrealistic"

## 6. Generate Images
This loop processes all images in `baseline_images`.

In [ ]:
from PIL import Image
import numpy as np

# Settings
HEALING_RATIO = 0.8  # 80% healing, 20% worsening

total_images = len(baseline_images)
num_healing = int(total_images * HEALING_RATIO)

print(f"Generating {num_healing} healing sequences and {total_images - num_healing} worsening sequences.")

for idx, img_path in enumerate(baseline_images):
    wound_id = f"wound_{idx:03d}"
    # Determine sequence type
    is_healing = idx < num_healing
    progression_type = "healing" if is_healing else "worsening"
    prompts = HEALING_PROMPTS if is_healing else WORSENING_PROMPTS
    
    print(f"[{idx+1}/{total_images}] Processing {img_path.name} -> {wound_id} ({progression_type})...")
    
    # Setup output folder for this wound
    sequence_dir = output_dir / wound_id
    sequence_dir.mkdir(exist_ok=True)
    
    # Load and resize baseline
    baseline_img = Image.open(img_path).convert("RGB")
    baseline_img = baseline_img.resize((512, 512))
    baseline_img.save(sequence_dir / "day_0.png")
    
    # Generate timepoints
    for timepoint, prompt in prompts.items():
        gen_image = pipe(
            prompt=prompt,
            image=baseline_img,
            strength=0.30,      # Consistency factor (lower = more like original)
            guidance_scale=7.5, # Adherence to prompt
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=30
        ).images[0]
        
        gen_image.save(sequence_dir / f"{timepoint}.png")
    
print("\n✨ Generation Complete!")

## 7. Zip and Download
Compress the results into a single file for easy download.

In [ ]:
import shutil

zip_filename = "synthetic_wounds"
shutil.make_archive(zip_filename, 'zip', output_dir)

print(f"✅ Zipped results to: {zip_filename}.zip")
print("You can now download this file from the File Browser on the left.")